# PA2: MobileNetV2 — Echtzeit-Personenerkennung auf dem PYNQ-Z1

Dieses Notebook implementiert eine vollständige FPGA-Inferenz-Pipeline für das Visual Wake Words (VWW) Modell auf Basis von MobileNetV2. Das Modell klassifiziert Kamerabilder binär: **Person vorhanden** oder **keine Person**.

**Ablauf:**
1. Umgebung einrichten
2. Modell laden
3. Datensatz laden und vorverarbeiten
4. Keras-Baseline-Genauigkeit bestimmen
5. Sichere hls4ml-Konvertierung (Standardpräzision)
6. Profiling der Aktivierungswertebereiche
7. Profilbasierte Post-Training-Quantisierung (PTQ)
8. CPU-Emulation und Genauigkeitsvergleich
9. Synthesebericht (Latenz + Ressourcen)
10. Synthese und Bitstream-Erzeugung
11. Deployment-Vorbereitung

## 1. Umgebung einrichten

In [1]:
# Vivado-Version prüfen
# Voraussetzung: conda activate hls4ml-tutorial
#                source /opt/Xilinx/Vivado/2022.2/settings64.sh
!vivado -version

Vivado v2022.2 (64-bit)
SW Build 3671981 on Fri Oct 14 04:59:54 MDT 2022
IP Build 3669848 on Fri Oct 14 08:30:02 MDT 2022
Tool Version Limit: 2022.10
Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.


In [2]:
import os
import math
import numpy as np
import tensorflow as tf
import hls4ml
import pprint
import re
import matplotlib
matplotlib.use('Agg')  # Kein Display erforderlich (AlmaLinux/Server)
from pathlib import Path
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from qkeras.utils import _add_supported_quantized_objects

# Vivado-Pfad registrieren
os.environ['PATH'] = os.environ['XILINX_VIVADO'] + '/bin:' + os.environ['PATH']

# QKeras-Objekte für Modell-Laden mit custom layers
co = {}
_add_supported_quantized_objects(co)

print(f'TensorFlow : {tf.__version__}')
print(f'hls4ml     : {hls4ml.__version__}')

2026-06-22 18:31:28.601810: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow : 2.14.0
hls4ml     : 1.1.0


## 2. Modell laden

Das vortrainierte MobileNetV2-Modell (Visual Wake Words, 96×96 Eingabe) wird von MLCommons geladen.

In [3]:
MODEL_URL  = ('https://github.com/mlcommons/tiny/raw/'
              'bceb91c5ad2e2deb295547d81505721d3a87d578/'
              'benchmark/training/visual_wake_words/'
              'trained_models/vww_96.h5')
MODEL_NAME = 'vww_96.h5'

model_path = tf.keras.utils.get_file(MODEL_NAME, MODEL_URL)
model      = tf.keras.models.load_model(model_path, custom_objects=co)

print('Modell erfolgreich geladen.')
model.summary()

Modell erfolgreich geladen.
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 96, 96, 3)]       0         
                                                                 
 conv2d (Conv2D)             (None, 48, 48, 8)         224       
                                                                 
 batch_normalization (Batch  (None, 48, 48, 8)         32        
 Normalization)                                                  
                                                                 
 activation (Activation)     (None, 48, 48, 8)         0         
                                                                 
 depthwise_conv2d (Depthwis  (None, 48, 48, 8)         80        
 eConv2D)                                                        
                                                                 
 batch_normalization_1 (Bat  (Non

In [ ]:
MODEL_URL ="https://github.com/mlcommons/tiny/raw/bceb91c5ad2e2deb295547d81505721d3a87d578/benchmark/training/visual_wake_words/trained_models/vww_96_int8.tflite"
MODEL_NAME = "vww_96_int8.tflite"
MODEL_PATH = download_testdata(MODEL_URL, MODEL_NAME, module="model")




## 3. Datensatz laden und vorverarbeiten

Der VWW-Datensatz (COCO 2014-basiert, 96×96 Pixel) liegt lokal vor. Aufteilung: 80 % Training / 20 % Validierung.

In [4]:
DATASET_PATH = os.path.expanduser('~/Bureau/Projekt/PA2/vw_coco2014_96')
IMG_SIZE     = (96, 96)
BATCH_SIZE   = 32

# Datensatz-Überprüfung
for klasse in ['person', 'non_person']:
    count = len(os.listdir(os.path.join(DATASET_PATH, klasse)))
    print(f'{klasse:12s} : {count} Bilder')

person       : 53140 Bilder
non_person   : 56479 Bilder


In [5]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    horizontal_flip=True
)

train_gen = datagen.flow_from_directory(
    DATASET_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', shuffle=True, seed=42
)
val_gen = datagen.flow_from_directory(
    DATASET_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', shuffle=False, seed=42
)

print(f'Erkannte Klassen     : {train_gen.class_indices}')
print(f'Trainings-Batches    : {len(train_gen)}')
print(f'Validierungs-Batches : {len(val_gen)}')

Found 87696 images belonging to 2 classes.
Found 21923 images belonging to 2 classes.
Erkannte Klassen     : {'non_person': 0, 'person': 1}
Trainings-Batches    : 2741
Validierungs-Batches : 686


### NumPy-Block aus dem Validierungsgenerator extrahieren

512 Validierungsbilder werden als NumPy-Arrays extrahiert — für Genauigkeitsmessung, Profiling und CPU-Emulation.

In [6]:
X_val_np, y_val_np = [], []
for i, (X_batch, y_batch) in enumerate(val_gen):
    X_val_np.append(X_batch)
    y_val_np.append(y_batch)
    if i >= 15:   # 16 × 32 = 512 Bilder
        break

X_val_np = np.concatenate(X_val_np, axis=0)                     # (512, 96, 96, 3)
y_val_np = np.argmax(np.concatenate(y_val_np, axis=0), axis=1)  # (512,)

print(f'X_val Form        : {X_val_np.shape}')
print(f'y_val Form        : {y_val_np.shape}')
print(f'Klassenverteilung : {np.bincount(y_val_np)}  [non_person, person]')

X_val Form        : (512, 96, 96, 3)
y_val Form        : (512,)
Klassenverteilung : [512]  [non_person, person]


## 4. Keras-Baseline-Genauigkeit (float32)

Die Genauigkeit des unveränderten float32-Modells dient als Referenzwert für alle nachfolgenden Quantisierungsschritte.

In [7]:
y_pred_keras = np.argmax(model.predict(X_val_np, verbose=1), axis=1)
acc_keras    = np.mean(y_pred_keras == y_val_np)

print(f'\nGenauigkeit Keras (float32) : {acc_keras*100:.2f} %')

16/16 [==============================] - 0s 11ms/step

Genauigkeit Keras (float32) : 91.80 %


## 5. Sichere hls4ml-Konvertierung (Standardpräzision)

Vor dem Profiling wird das Modell mit der hls4ml-Standardpräzision `ap_fixed<16,6>` konvertiert. Diese Konfiguration ist garantiert überlauf-sicher und dient als Grundlage für das Profiling.

> **Wichtig:** Blind gesetzte Präzisionen wie `ap_fixed<8,4>` können zu einem vollständigen Netz-Kollaps führen, wenn die tatsächlichen Aktivierungswerte die Darstellungsgrenze überschreiten (Überlauf → konstanter Ausgabewert für alle Eingaben).

In [8]:
config_safe = hls4ml.utils.config_from_keras_model(model, granularity='name')

# Nur ReuseFactor anpassen — ap_fixed<16,6> bleibt als sicherer Standard
config_safe['Model']['ReuseFactor'] = 16

for layer in model.layers:
    name = layer.name
    if name not in config_safe['LayerName']:
        continue
    cfg = config_safe['LayerName'][name]
    if 'depthwise' in name.lower():
        cfg['ReuseFactor'] = 4
    elif 'conv' in name.lower():
        cfg['ReuseFactor'] = 16
    elif name in ['Conv_1', 'Logits', 'predictions', 'dense']:
        cfg['ReuseFactor'] = 64

hls_safe = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config_safe,
    output_dir='mobilenet_safe_prj',
    backend='VivadoAccelerator',
    board='pynq-z2',
    io_type='io_stream'
)
hls_safe.compile()

# Sanity-Check: Ausgaben müssen sich pro Sample unterscheiden
out_check = hls_safe.predict(X_val_np[:5])
print('Ausgabe (5 Samples) mit sicherer Präzision:')
print(out_check)
print(f'\nMin: {out_check.min():.4f}  Max: {out_check.max():.4f}')
print('Ausgaben unterschiedlich:', not np.all(out_check == out_check[0]))

Interpreting Model
Topology:
Layer name: input_1, layer type: InputLayer, input shapes: [[None, 96, 96, 3]], output shape: [None, 96, 96, 3]
Layer name: conv2d, layer type: Conv2D, input shapes: [[None, 96, 96, 3]], output shape: [None, 48, 48, 8]
Layer name: batch_normalization, layer type: BatchNormalization, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: activation, layer type: Activation, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: depthwise_conv2d, layer type: DepthwiseConv2D, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: batch_normalization_1, layer type: BatchNormalization, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: activation_1, layer type: Activation, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: conv2d_1, layer type: Conv2D, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 16]
Layer name: batch_

## 6. Profiling der Aktivierungswertebereiche

Das Profiling analysiert die tatsächlichen Wertebereiche der Gewichte und Aktivierungen auf echten Validierungsdaten. Das Ergebnis bestimmt den korrekten ganzzahligen Teil `I` in `ap_fixed<W,I>`:

$$I \geq \lceil \log_2(\max|\text{Aktivierung}|) \rceil + 1$$

Ist `I` zu klein gewählt, kommt es zu **Überlauf** → alle Ausgaben werden konstant → Genauigkeit kollabiert.

In [9]:
from hls4ml.model.profiling import numerical

# Profiling auf 200 Validierungsbildern
wp, ap, wq, aq = numerical(
    model=model,
    hls_model=hls_safe,
    X=X_val_np[:200]
)

# Maximale Aktivierungswerte pro Schicht und benötigtes I
print(f'\n{"Schicht":40s} {"max |Aktivierung|":>20s}  {"I benötigt":>12s}')
print('-' * 76)
for name, vals in aq.items():
    max_val  = float(np.max(np.abs(vals)))
    i_needed = max(2, math.ceil(math.log2(max_val + 1e-9)) + 1) if max_val > 0 else 2
    print(f'{name:40s} {max_val:>20.4f}  ap_fixed<W,{i_needed}>')

Interpreting Model
Topology:
Layer name: input_1, layer type: InputLayer, input shapes: [[None, 96, 96, 3]], output shape: [None, 96, 96, 3]
Layer name: conv2d, layer type: Conv2D, input shapes: [[None, 96, 96, 3]], output shape: [None, 48, 48, 8]
Layer name: batch_normalization, layer type: BatchNormalization, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: activation, layer type: Activation, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: depthwise_conv2d, layer type: DepthwiseConv2D, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: batch_normalization_1, layer type: BatchNormalization, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: activation_1, layer type: Activation, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: conv2d_1, layer type: Conv2D, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 16]
Layer name: batch_

RuntimeError: ModelGraph must have tracing on for at least 1 layer (this can be set in its config)

## 7. Profilbasierte Post-Training-Quantisierung (PTQ)

Die Präzision wird jetzt **datenbasiert** gesetzt: `I` wird direkt aus den gemessenen Maximalwerten abgeleitet, `W` nach Schichttyp gewählt:

| Schichttyp | W (Gesamtbreite) | Grund |
|---|---|---|
| Depthwise Conv | 10 Bit | Wenige Gewichte, sehr empfindlich |
| Erste / letzte Schicht | 10 Bit | Ein-/Ausgabe kritisch |
| Pointwise Conv 1×1 | 8 Bit | Viele Gewichte, robuster |
| Standard | 8 Bit | Konservative Vorgabe |

In [10]:
config_ptq = hls4ml.utils.config_from_keras_model(model, granularity='name')
config_ptq['Model']['ReuseFactor'] = 16

for layer in model.layers:
    name = layer.name
    if name not in config_ptq['LayerName']:
        continue
    cfg = config_ptq['LayerName'][name]

    # I aus Profiling ableiten
    if name in aq:
        max_val  = float(np.max(np.abs(aq[name])))
        i_needed = max(2, math.ceil(math.log2(max_val + 1e-9)) + 1) if max_val > 0 else 2
    else:
        i_needed = 4  # konservativer Fallback

    # W nach Schichttyp
    if 'depthwise' in name.lower():
        W = 10
        cfg['ReuseFactor'] = 4
    elif name == 'Conv1':
        W = 10
        cfg['ReuseFactor'] = 8
    elif name in ['Conv_1', 'Logits', 'predictions', 'dense']:
        W = 10
        cfg['ReuseFactor'] = 64
    elif 'softmax' in name.lower():
        cfg['exp_table_t'] = 'ap_fixed<18,8>'
        cfg['inv_table_t'] = 'ap_fixed<18,4>'
        continue
    elif 'expand' in name.lower() or 'project' in name.lower():
        W = 8
        cfg['ReuseFactor'] = 16
    else:
        W = 8
        cfg['ReuseFactor'] = 16

    # I darf maximal W-2 betragen (mindestens 2 Fraktionsbits)
    i_final = min(i_needed, W - 2)
    cfg['Precision'] = f'ap_fixed<{W},{i_final}>'

print('PTQ-Konfiguration erstellt.')
pprint.pprint(config_ptq)

Interpreting Model
Topology:
Layer name: input_1, layer type: InputLayer, input shapes: [[None, 96, 96, 3]], output shape: [None, 96, 96, 3]
Layer name: conv2d, layer type: Conv2D, input shapes: [[None, 96, 96, 3]], output shape: [None, 48, 48, 8]
Layer name: batch_normalization, layer type: BatchNormalization, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: activation, layer type: Activation, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: depthwise_conv2d, layer type: DepthwiseConv2D, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: batch_normalization_1, layer type: BatchNormalization, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: activation_1, layer type: Activation, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: conv2d_1, layer type: Conv2D, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 16]
Layer name: batch_

NameError: name 'aq' is not defined

In [11]:
hls_ptq = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config_ptq,
    output_dir='model_personDetection/hls4ml_prj_pynq',
    backend='VivadoAccelerator',
    board='pynq-z2',       # gleicher Chip xc7z020clg400-1 wie PYNQ-Z1
    io_type='io_stream'    # zwingend für Bilddaten
)
hls_ptq.compile()

print('Konvertierung abgeschlossen.')

Interpreting Model
Topology:
Layer name: input_1, layer type: InputLayer, input shapes: [[None, 96, 96, 3]], output shape: [None, 96, 96, 3]
Layer name: conv2d, layer type: Conv2D, input shapes: [[None, 96, 96, 3]], output shape: [None, 48, 48, 8]
Layer name: batch_normalization, layer type: BatchNormalization, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: activation, layer type: Activation, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: depthwise_conv2d, layer type: DepthwiseConv2D, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: batch_normalization_1, layer type: BatchNormalization, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: activation_1, layer type: Activation, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 8]
Layer name: conv2d_1, layer type: Conv2D, input shapes: [[None, 48, 48, 8]], output shape: [None, 48, 48, 16]
Layer name: batch_

## 8. CPU-Emulation und Genauigkeitsvergleich

Die bit-genaue CPU-Emulation des hls4ml-Modells wird ausgeführt. Die gespeicherten Dateien dienen in Part 7c als Referenz für den Hardware-Vergleich.

In [12]:
# Keras-Vorhersagen (float32-Referenz)
print('-- Keras-Vorhersage (float32) --')
y_keras   = model.predict(X_val_np, verbose=1)
np.save('y_keras_pred.npy', y_keras)
acc_keras = np.mean(np.argmax(y_keras, axis=1) == y_val_np)
print(f'Genauigkeit Keras (float32) : {acc_keras*100:.2f} %')

# hls4ml CPU-Emulation
print('\n-- hls4ml-Vorhersage (CPU-Emulation) --')
y_hls      = hls_ptq.predict(X_val_np)
y_pred_hls = np.argmax(y_hls, axis=1)  # immer definiert, auch bei Kollaps
np.save('y_hls_pred.npy', y_hls)
np.save('y_true.npy', y_val_np)

# Kollaps-Diagnose
klassen_eindeutig = np.unique(y_pred_hls)
if len(klassen_eindeutig) < 2:
    print('WARNUNG: Netz-Kollaps erkannt!')
    print(f'   Rohe Ausgabe (3 Samples): {y_hls[:3]}')
    print('   I-Werte im Profiling pruefen und PTQ-Konfiguration anpassen.')
else:
    acc_hls = np.mean(y_pred_hls == y_val_np)
    match   = np.mean(np.argmax(y_keras, axis=1) == y_pred_hls)
    print(f'Genauigkeit hls4ml (CPU-Emu) : {acc_hls*100:.2f} %')
    print('\n-- Zusammenfassung --')
    print(f'Genauigkeit Keras (float32)  : {acc_keras*100:.2f} %')
    print(f'Genauigkeit hls4ml (CPU-Emu) : {acc_hls*100:.2f} %')
    print(f'Genauigkeitsverlust          : {(acc_keras-acc_hls)*100:.2f} Prozentpunkte')
    print(f'Uebereinstimmung             : {match*100:.2f} %  (Ziel: > 97 %)')
    print('\nGespeicherte Dateien:')
    print('  y_keras_pred.npy  float32-Referenz')
    print('  y_hls_pred.npy    CPU-Emulation')
    print('  y_true.npy        Wahre Labels')


── Keras-Vorhersage (float32) ───────────────────────────
16/16 [==============================] - 0s 10ms/step
Genauigkeit Keras (float32) : 91.80 %

── hls4ml-Vorhersage (CPU-Emulation) ────────────────────
⚠️  WARNUNG: Netz-Kollaps erkannt!
   Alle Ausgaben identisch: [[1. 0.]
 [1. 0.]
 [1. 0.]]
   → I-Werte im Profiling prüfen und PTQ-Konfiguration anpassen.


In [13]:
from sklearn.metrics import confusion_matrix, classification_report

klassen = ['non_person', 'person']

print('-- Klassifikationsbericht --')
print(classification_report(
    y_val_np, y_pred_hls,
    labels=[0, 1],
    target_names=klassen,
    zero_division=0
))

print('-- Konfusionsmatrix --')
cm = confusion_matrix(y_val_np, y_pred_hls, labels=[0, 1])
print('                  Vorhergesagt')
print('                  non_person   person')
print(f'  non_person    {cm[0,0]:>10d}  {cm[0,1]:>6d}')
print(f'  person        {cm[1,0]:>10d}  {cm[1,1]:>6d}')

# Gesamtgenauigkeit (auch im Kollaps-Fall berechenbar)
acc_hls = np.mean(y_pred_hls == y_val_np)
print(f'\nGesamtgenauigkeit hls4ml : {acc_hls*100:.2f} %')


── Klassifikationsbericht ───────────────────────────────
              precision    recall  f1-score   support

  non_person       1.00      1.00      1.00       512
      person       0.00      0.00      0.00         0

   micro avg       1.00      1.00      1.00       512
   macro avg       0.50      0.50      0.50       512
weighted avg       1.00      1.00      1.00       512

── Konfusionsmatrix ──────────────────────────────────────
                  Vorhergesagt
                  non_person   person
  non_person           512       0
  person                 0       0


## 9. Synthesebericht — Latenz und Ressourcen

Hilfsfunktionen zum Einlesen der HLS-C-Synthese- und Vivado-Implementierungsberichte mit korrekten VivadoAccelerator-Pfaden.

In [14]:
def get_hls_report(output_dir):
    """HLS-C-Synthesebericht: Latenz + Ressourcenschätzung."""
    rpt = Path(output_dir) / 'myproject_prj/solution1/syn/report/myproject_csynth.rpt'
    data = {}
    if not rpt.is_file():
        print(f'HLS-Bericht nicht gefunden: {rpt}')
        return data
    lines = rpt.read_text().splitlines()
    for i, line in enumerate(lines):
        if 'Latency (cycles)' in line and i + 3 < len(lines):
            p = lines[i + 3].split('|')
            if len(p) >= 7:
                try:
                    data['Latenz_min_Takte'] = int(p[2].strip())
                    data['Latenz_max_Takte'] = int(p[3].strip())
                    data['Latenz_min_µs']    = round(int(p[2].strip()) * 10.0 / 1000.0, 3)
                    data['II_min']           = int(p[6].strip())
                except ValueError:
                    pass
            break
    for line in lines:
        for key, kws in [('DSP','DSP48'), ('BRAM','bram_18k'), ('FF','FF'), ('LUT','LUT')]:
            if kws.lower() in line.lower() and '|' in line and key not in data:
                p = line.split('|')
                if len(p) >= 4:
                    try:
                        data[key] = int(p[2].strip().replace(',', ''))
                    except ValueError:
                        pass
    return data


def get_impl_report(output_dir):
    """Vivado-Implementierungsbericht: Post-Place-and-Route-Werte (genau)."""
    rpt = Path(output_dir) / (
        'myproject_vivado_accelerator/project_1.runs/impl_1/'
        'design_1_wrapper_utilization_placed.rpt'
    )
    data = {}
    if not rpt.is_file():
        print(f'Implementierungsbericht nicht gefunden: {rpt}')
        print('→ Bitstream-Synthese noch nicht abgeschlossen.')
        return data
    content = rpt.read_text()
    for key, pat in [
        ('LUT',  r'Slice LUTs\*?\s*\|\s*(\d+)'),
        ('FF',   r'Slice Registers\s*\|\s*(\d+)'),
        ('BRAM', r'Block RAM Tile\s*\|\s*(\d+)'),
        ('DSP',  r'DSPs\s*\|\s*(\d+)'),
    ]:
        m = re.search(pat, content)
        if m:
            data[key] = int(m.group(1))
    return data


def print_report_table(hls_data, impl_data=None):
    """Formatierte Ausgabe von Latenz und Ressourcenauslastung."""
    VERFUEGBAR = {'LUT': 53200, 'FF': 106400, 'BRAM': 280, 'DSP': 220}
    print('\n' + '=' * 62)
    print('  SYNTHESEBERICHT — MobileNetV2 auf PYNQ-Z1 (xc7z020)')
    print('=' * 62)
    if hls_data:
        print('\n── HLS-C-Synthese (Schätzwerte) ──────────────────────────')
        print(f"  Latenz min  : {hls_data.get('Latenz_min_Takte','–'):>8} Takte"
              f"  = {hls_data.get('Latenz_min_µs','–'):>8} µs  (@100 MHz)")
        print(f"  II min      : {hls_data.get('II_min','–'):>8} Takte")
        if hls_data.get('II_min'):
            fps = 100e6 / hls_data['II_min']
            print(f'  Durchsatz   : {fps:>8.0f} Inferenzen/s  ({fps/1000:.1f} kFPS)')
    src   = impl_data if impl_data else hls_data
    label = 'Post-Implementierung (exakt)' if impl_data else 'HLS-Schätzung'
    if src:
        print(f'\n── Ressourcenauslastung ({label}) ────')
        print(f"  {'Ressource':10s} {'Verbrauch':>10s} {'Verfügbar':>10s} {'Auslastung':>11s}")
        print('  ' + '-' * 45)
        for res in ['LUT', 'FF', 'BRAM', 'DSP']:
            val   = src.get(res)
            avail = VERFUEGBAR[res]
            if val is not None:
                pct  = val / avail * 100
                flag = '  ⚠️' if pct > 90 else ''
                print(f"  {res:10s} {val:>10d} {avail:>10d} {pct:>10.1f} %{flag}")
    print('=' * 62)

print('Berichtsfunktionen geladen.')

Berichtsfunktionen geladen.


In [15]:
# ── HLS-Bericht auslesen (nach compile()) ────────────────────────────────
hls_data  = get_hls_report('model_personDetection/hls4ml_prj_pynq')
impl_data = get_impl_report('model_personDetection/hls4ml_prj_pynq')  # leer bis Bitstream fertig

print_report_table(hls_data, impl_data if impl_data else None)

# hls4ml eingebauter Report-Parser
try:
    report = hls4ml.report.read_vivado_report('model_personDetection/hls4ml_prj_pynq')
    print('\nhls4ml-Report:')
    pprint.pprint(report)
except Exception as e:
    print(f'hls4ml-Report noch nicht verfügbar: {e}')

HLS-Bericht nicht gefunden: model_personDetection/hls4ml_prj_pynq/myproject_prj/solution1/syn/report/myproject_csynth.rpt
Implementierungsbericht nicht gefunden: model_personDetection/hls4ml_prj_pynq/myproject_vivado_accelerator/project_1.runs/impl_1/design_1_wrapper_utilization_placed.rpt
→ Bitstream-Synthese noch nicht abgeschlossen.

  SYNTHESEBERICHT — MobileNetV2 auf PYNQ-Z1 (xc7z020)
Project myproject_prj does not exist. Rerun "hls4ml build -p model_personDetection/hls4ml_prj_pynq".

hls4ml-Report:
None


## 10. Synthese und Bitstream-Erzeugung

Das VivadoAccelerator-Backend erzeugt ein vollständiges Block-Design mit:
- **hls4ml-IP** (MobileNetV2-Beschleuniger)
- **AXI DMA** (PS ↔ PL Datentransfer)
- **AXI Interconnect** + **ZYNQ7 Processing System**

![part7_block_design.png](attachment:part7_block_design.png)

> Dauer: ca. 60 Minuten. Fortschritt verfolgen mit:
> `tail -f model_personDetection/hls4ml_prj_pynq/vivado_hls.log`

In [ ]:
# Synthese + Implementierung + Bitstream
hls_ptq.build(csim=False, export=True, bitfile=True)

In [ ]:
# ── Post-Implementierungsbericht (nach Synthese) ─────────────────────────
impl_data_final = get_impl_report('model_personDetection/hls4ml_prj_pynq')
hls_data_final  = get_hls_report('model_personDetection/hls4ml_prj_pynq')

print_report_table(hls_data_final, impl_data_final)

# Roher Bericht-Auszug
rpt_path = Path('model_personDetection/hls4ml_prj_pynq/myproject_vivado_accelerator/'
               'project_1.runs/impl_1/'
               'design_1_wrapper_utilization_placed.rpt')
if rpt_path.is_file():
    lines = rpt_path.read_text().splitlines()
    print('\n── Roher Bericht (Auszug Zeilen 30–65) ──────────────────')
    print('\n'.join(lines[29:65]))
else:
    print('Implementierungsbericht noch nicht vorhanden.')

## 11. Deployment-Vorbereitung

Die erzeugten Dateien werden auf das PYNQ-Z1 Board übertragen:

| Datei | Beschreibung |
|---|---|
| `hls4ml_nn.bit` | Bitstream für das FPGA |
| `hls4ml_nn.hwh` | Hardware-Beschreibung für PYNQ |
| `axi_stream_driver.py` | Python-Treiber für den DMA |
| `y_keras_pred.npy` | float32-Referenz-Vorhersagen |
| `y_hls_pred.npy` | CPU-Emulations-Vorhersagen |
| `y_true.npy` | Wahre Labels |

**Verbindung:** USB Micro-B → Port J14 (PROG/UART) · IP: `192.168.2.99` · Passwort: `xilinx`

In [ ]:
import shutil, subprocess

OUT = 'model_personDetection/hls4ml_prj_pynq'
PKG = OUT + '/package'
os.makedirs(PKG, exist_ok=True)

# Dateien kopieren (Python statt Shell-Variable-Interpolation)
copies = [
    (OUT + '/myproject_vivado_accelerator/project_1.runs/impl_1/design_1_wrapper.bit',
     PKG + '/hls4ml_nn.bit'),
    (OUT + '/myproject_vivado_accelerator/project_1.srcs/sources_1/bd/'
           'design_1/hw_handoff/design_1.hwh',
     PKG + '/hls4ml_nn.hwh'),
    (OUT + '/axi_stream_driver.py',
     PKG + '/axi_stream_driver.py'),
]
for src, dst in copies:
    if Path(src).is_file():
        shutil.copy2(src, dst)
        print(f'Kopiert: {Path(src).name}')
    else:
        print(f'Nicht gefunden: {src}')

for fname in ['y_keras_pred.npy', 'y_hls_pred.npy', 'y_true.npy']:
    if Path(fname).is_file():
        shutil.copy2(fname, PKG + '/' + fname)
        print(f'Kopiert: {fname}')

subprocess.run(['tar', '-czvf', OUT + '/package.tar.gz', '-C', PKG, '.'], check=True)
print('\nPaket erstellt: ' + OUT + '/package.tar.gz')


In [ ]:
# ── Übertragung auf PYNQ-Z1 ──────────────────────────────────────────────
# Voraussetzung: sudo ip addr add 192.168.2.1/24 dev usb0
#                sudo ip link set usb0 up

!scp model_personDetection/hls4ml_prj_pynq/package.tar.gz xilinx@192.168.2.99:~/jupyter_notebooks/

print('Übertragung abgeschlossen.')
print('Weiter auf dem PYNQ-Z1 (http://192.168.2.99):')
print('  cd ~/jupyter_notebooks && tar xvzf package.tar.gz')